# Graduate Students and Generative AI

### Reproducible analysis of the 2025 CU Boulder gradSERU sample

**Leadership question:** How well have faculty discussion and program guidance kept pace
with graduate students' adoption and uses of generative AI?

This notebook builds a narrow analytical layer from the 289-column survey export, validates
the relevant denominators, and develops a small set of information-rich Altair figures.

#### Interpretation guardrails

- The file is a **single 2025 survey cross-section**. `YEAR` and `TERM1` describe entry
  cohort/term; they are not repeated survey waves.
- `LEVEL_GRAD` contains codes 1, 3, and 4, but their labels are not supplied. The analysis
  treats them as categorical codes and does not invent degree labels.
- The export uses both missing values and `-1`. For these analyses, `-1` is treated as
  unavailable/nonresponse, never as a substantive survey response.
- Estimates are unweighted respondent summaries because no survey-weight field was supplied.
  They should not be presented as population estimates for all CU Boulder graduate students.

## 1. Setup and source data

The notebook needs the supplied Excel workbook either beside this notebook or in an
`upload/` subdirectory. The compact label dictionaries below are manually verified against
the survey instrument, including the corrected `_8` and `_9` AI-purpose fields.

In [1]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

CU_GOLD = "#D8B04D"
CU_BLACK = "#000000"
CU_DARK_GRAY = "#333537"
CU_LIGHT_GRAY = "#D9D9D9"
CU_BLUE = "#4C78A8"
PAPER = "#E4E4E4"


def style_chart(chart):
    return (
        chart
        .configure_view(
            stroke=None,  # removes top/right frame without removing axes
        )
        .configure_title(
            anchor="middle",
            align="center",
            frame="group",
            font="Arial",
            fontSize=20,
            fontWeight="bolder",
            color=CU_BLACK,
            subtitleFont="Arial",
            subtitleFontSize=16,
            subtitleColor=CU_DARK_GRAY,
            subtitlePadding=6,
            offset=12,
        )
        .configure_legend(
            orient="top",
            direction="horizontal",
            title=None,
            labelFont="Arial",
            labelFontSize=12,
            symbolSize=150,
            offset=12,
            strokeColor=CU_DARK_GRAY,
            strokeWidth=0.5,
            fillColor=PAPER,
            padding=4,
        )
        .configure_axis(
            domain=True,              # retain bottom and left axis lines
            domainColor=CU_BLACK,
            domainWidth=1,
            ticks=True,
            tickColor=CU_DARK_GRAY,
            tickWidth=0.5,
            labelFont="Arial",
            labelFontSize=12,
            labelFontWeight=600,
            titleFont="Arial",
            titleFontSize=14,
            titleFontWeight="bold",
            gridColor=CU_LIGHT_GRAY,
            gridOpacity=0.33,
        )
        .configure_axisX(
            orient="bottom",
        )
        .configure_axisY(
            orient="left",
            labelLimit=500,
            labelPadding=10,
        )
    )


_ = alt.data_transformers.disable_max_rows()

In [2]:
DATA_FILENAME = "CU Boulder SERU dataset_sample.xlsx"
DATA_CANDIDATES = [
    Path(DATA_FILENAME),
    Path("upload") / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Place {DATA_FILENAME!r} beside this notebook or in an upload/ subdirectory."
    )

FREQUENCY = "GS1301_GSGAIDAY"
FREQ_ORDER = [
    "Never",
    "Several times per year",
    "Several times per month",
    "Several times per week",
    "Daily",
]
FREQ_COLORS = ["#A9ADB0", "#E2D9C2", CU_GOLD, "#8A7950", CU_BLACK]
WEEKLY_LEVELS = ["Several times per week", "Daily"]
USAGE_TIER_ORDER = ["Never", "Occasional", "Frequent"]

PURPOSE_LABELS = {
    "GS1302_GSGGAI_1": "Research a topic",
    "GS1302_GSGGAI_4": "Brainstorm ideas",
    "GS1302_GSGGAI_5": "Draft a paper or presentation",
    "GS1302_GSGGAI_6": "Revise a paper or presentation",
    "GS1302_GSGGAI_7": "Draft assignment responses",
    "GS1302_GSGGAI_8": "Revise assignment responses",
    "GS1302_GSGGAI_9": "Generate programming code",
    "GS1302_GSGGAI_10": "Revise or debug code",
    "GS1302_GSGGAI_11": "Prepare for exams",
    "GS1302_GSGGAI_12": "Translate academic/professional work",
    "GS1302_GSGGAI_999": "Other",
}
PURPOSE_SHORT = {
    "GS1302_GSGGAI_1": "Research",
    "GS1302_GSGGAI_4": "Brainstorm",
    "GS1302_GSGGAI_5": "Draft paper",
    "GS1302_GSGGAI_6": "Revise paper",
    "GS1302_GSGGAI_7": "Draft answers",
    "GS1302_GSGGAI_8": "Revise answers",
    "GS1302_GSGGAI_9": "Generate code",
    "GS1302_GSGGAI_10": "Debug code",
    "GS1302_GSGGAI_11": "Exam prep",
    "GS1302_GSGGAI_12": "Translation",
    "GS1302_GSGGAI_999": "Other",
}

ATTITUDE_LABELS = {
    "GS1303_GSGAIPDC": "Professors discussed appropriate coursework use",
    "GS1303_GSGAIAUC": "I understand appropriate coursework use",
    "GS1303_GSGAIGPT": "Program guidance for teaching",
    "GS1303_GSGAIGURW": "Program guidance for research and writing",
    "GS1303_GSGAIGR": "I understand how AI generates responses",
    "GS1303_GSGAIEPDR": "I can create effective prompts",
}
LIKERT_ORDER = ["Strongly disagree", "Disagree", "Agree", "Strongly agree"]
LIKERT_COLORS = ["#7B2D26", "#C88F84", "#D9C99F", CU_GOLD]

COLLEGE_SHORT = {
    "College of Arts & Sciences": "Arts & Sciences",
    "College of Engr & Applied Sci": "Engineering",
    "College of Media, Comm & Info": "Media, Comm & Info",
    "College of Music": "Music",
    "Cross-College Programs": "Cross-College",
    "Leeds School of Business": "Business",
    "School of Education": "Education",
}

PURPOSE_COLUMNS = list(PURPOSE_LABELS)
ATTITUDE_COLUMNS = list(ATTITUDE_LABELS)
INSTITUTION_COLUMNS = [
    "FINISHED",
    "PROGRESS",
    "COLLEGE_CODE1",
    "COLLEGE_NAME1",
    "PROGRAM_CODE1",
    "PROGRAM_TEXT1",
    "LEVEL_GRAD",
    "YEAR",
    "TERM1",
]
REQUIRED_COLUMNS = INSTITUTION_COLUMNS + [FREQUENCY] + PURPOSE_COLUMNS + ATTITUDE_COLUMNS

In [3]:
raw_survey = pd.read_excel(DATA_PATH, sheet_name="CU.BOULDER_GRAD25_Labels")

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(raw_survey.columns))
if missing_columns:
    raise KeyError(f"Required columns are missing: {missing_columns}")
    # Preserve the full 289-column source. `survey` subsets that with REQUIRED_COLUMNS which contains the AI use questions
    # and Respondent identifiers:                                                                      
    #    ['FINISHED', 'PROGRESS', 'COLLEGE_CODE1', 'COLLEGE_NAME1',                 
    #     'PROGRAM_CODE1', 'PROGRAM_TEXT1', 'LEVEL_GRAD', 'YEAR', 'TERM1']

survey = raw_survey.loc[:, REQUIRED_COLUMNS].copy()
survey.insert(0, "respondent_id", np.arange(len(survey)))
survey = survey.replace({-1: pd.NA, "-1": pd.NA})
survey = survey.rename(columns={"YEAR": "ENTRY_YEAR", "TERM1": "ENTRY_TERM"})

valid_frequency = survey[FREQUENCY].isin(FREQ_ORDER)
ai_respondents = survey.loc[valid_frequency].copy()
ai_respondents["weekly_or_more"] = ai_respondents[FREQUENCY].isin(WEEKLY_LEVELS)
ai_respondents["usage_tier"] = ai_respondents[FREQUENCY].map(
    {
        "Never": "Never",
        "Several times per year": "Occasional",
        "Several times per month": "Occasional",
        "Several times per week": "Frequent",
        "Daily": "Frequent",
    }
)

    # Purposes were displayed only to AI users. Require a complete Checked/Not Checked battery.
purpose_eligible = ai_respondents[FREQUENCY].ne("Never") #  Respondents who said they never use AI did not see AI questions
purpose_complete = ai_respondents[PURPOSE_COLUMNS].isin( #  Omits NULL aka people who didn't get the AI questions
    ["Checked", "Not Checked"]
).all(axis=1)
purpose_base = ai_respondents.loc[purpose_eligible & purpose_complete].copy()

    # Subset the AI responders with use identifiers, use Frequency, and use purposes 
purpose_long = (
    purpose_base[
        ["respondent_id", "COLLEGE_NAME1", "LEVEL_GRAD", FREQUENCY] + PURPOSE_COLUMNS
    ]
    .melt(      # group identifiers and response values
        id_vars=["respondent_id", "COLLEGE_NAME1", "LEVEL_GRAD", FREQUENCY],
        value_vars=PURPOSE_COLUMNS,
        var_name="code",
        value_name="response",
    )
    .assign(
        Purpose=lambda frame: frame["code"].map(PURPOSE_LABELS),
        PurposeShort=lambda frame: frame["code"].map(PURPOSE_SHORT), # map each purpose code to its truncated version 
        Selected=lambda frame: frame["response"].eq("Checked"), # Created bool for whether or not the Purpose was sected or not
    )
)

# Each attitude item retains its own valid base because two items were conditional.
attitude_long = (
    ai_respondents[["respondent_id"] + ATTITUDE_COLUMNS]
    .melt(
        id_vars="respondent_id",
        value_vars=ATTITUDE_COLUMNS,
        var_name="code",
        value_name="Response",
    )
)
attitude_long = attitude_long.loc[attitude_long["Response"].isin(LIKERT_ORDER)].copy()
attitude_long["Item"] = attitude_long["code"].map(ATTITUDE_LABELS)

# Mutually exclusive states for the alluvial figure.
guidance_paths = ai_respondents.loc[
    ai_respondents["GS1303_GSGAIPDC"].isin(LIKERT_ORDER)
    & ai_respondents["GS1303_GSGAIAUC"].isin(LIKERT_ORDER),
    ["respondent_id", "usage_tier", "GS1303_GSGAIPDC", "GS1303_GSGAIAUC"],
].copy()
guidance_paths["Faculty discussion"] = np.where(
    guidance_paths["GS1303_GSGAIPDC"].isin(["Agree", "Strongly agree"]),
    "Discussed",
    "Not discussed",
)
guidance_paths["Appropriate-use understanding"] = np.where(
    guidance_paths["GS1303_GSGAIAUC"].isin(["Agree", "Strongly agree"]),
    "Understands",
    "Does not understand",
)

## 2. Data quality and analytical bases

Partial survey completions are retained when the relevant AI item is valid. This is an
item-level complete-case strategy: respondents who answered the AI module but stopped later
still contribute usable information. A finished-only sensitivity check is shown below.

In [4]:
quality_summary = pd.DataFrame(
    [
        ("Raw survey records", len(raw_survey), "All consenting records in supplied file"),
        (
            "Finished surveys",
            int(raw_survey["FINISHED"].eq(True).sum()),
            "Not required if the relevant AI item is valid",
        ),
        (
            "Valid AI-frequency responses",
            len(ai_respondents),
            "Excludes missing and sentinel -1",
        ),
        (
            "Complete purpose batteries",
            len(purpose_base),
            "AI users with all 11 items coded Checked/Not Checked",
        ),
        (
            "Guidance-path records",
            len(guidance_paths),
            "Valid frequency, faculty-discussion, and understanding responses",
        ),
    ],
    columns=["Analytical base", "n", "Definition"],
)

finished_ai = ai_respondents.loc[ai_respondents["FINISHED"].eq(True)]
sensitivity = pd.DataFrame(
    {
        "Sample": ["All valid AI-frequency responses", "Finished surveys only"],
        "n": [len(ai_respondents), len(finished_ai)],
        "Weekly or more": [
            ai_respondents["weekly_or_more"].mean(),
            finished_ai["weekly_or_more"].mean(),
        ],
    }
)

assert ai_respondents[FREQUENCY].isin(FREQ_ORDER).all()
assert purpose_base[FREQUENCY].ne("Never").all()
assert purpose_base[PURPOSE_COLUMNS].isin(["Checked", "Not Checked"]).all().all()
assert attitude_long["Response"].isin(LIKERT_ORDER).all()
assert set(ai_respondents["LEVEL_GRAD"].dropna().astype(int).unique()) == {1, 3, 4}

display(
    quality_summary.style.hide(axis="index")
    .format({"n": "{:,.0f}"})
    .set_caption("Validated analytical bases")
)
display(
    sensitivity.style.hide(axis="index")
    .format({"n": "{:,.0f}", "Weekly or more": "{:.1%}"})
    .set_caption("Completion-status sensitivity check")
)

Analytical base,n,Definition
Raw survey records,"1,387",All consenting records in supplied file
Finished surveys,"1,152",Not required if the relevant AI item is valid
Valid AI-frequency responses,"1,177",Excludes missing and sentinel -1
Complete purpose batteries,886,AI users with all 11 items coded Checked/Not Checked
Guidance-path records,"1,154","Valid frequency, faculty-discussion, and understanding responses"


Sample,n,Weekly or more
All valid AI-frequency responses,"1,177",28.1%
Finished surveys only,"1,143",28.0%


## 3. Leadership-facing figures

These figures are designed as candidate building blocks for a 3–5 slide presentation. Each
uses a validated base and makes its denominator available in the title, labels, or tooltip.

In [5]:
# Overall use: one compact distribution with two leadership-ready summary statistics.
frequency_summary = (
    ai_respondents[FREQUENCY]
    .value_counts()
    .reindex(FREQ_ORDER, fill_value=0)
    .rename_axis("Frequency")
    .reset_index(name="Respondents")
)
frequency_summary["Percent"] = 100 * frequency_summary["Respondents"] / len(ai_respondents)
frequency_summary["Metric"] = "Valid AI-frequency respondents"
frequency_summary["Label"] = frequency_summary["Percent"].map(lambda value: f"{value:.0f}%")
frequency_summary["Order"] = frequency_summary["Frequency"].map(
    {value: index for index, value in enumerate(FREQ_ORDER)}
)
frequency_summary["Start"] = frequency_summary["Percent"].cumsum() - frequency_summary["Percent"]
frequency_summary["End"] = frequency_summary["Percent"].cumsum()
frequency_summary["Midpoint"] = (frequency_summary["Start"] + frequency_summary["End"]) / 2

ever_used = 100 * ai_respondents[FREQUENCY].ne("Never").mean()
weekly_used = 100 * ai_respondents["weekly_or_more"].mean()

frequency_bar = (
    alt.Chart(frequency_summary)
    .mark_bar(height=54)
    .encode(
        x=alt.X("Start:Q", title="Share of respondents", axis=alt.Axis(format=".0f")),
        x2="End:Q",
        y=alt.Y("Metric:N", title=None, axis=alt.Axis(labels=False, ticks=False, domain=False)),
        color=alt.Color(
            "Frequency:N",
            sort=FREQ_ORDER,
            scale=alt.Scale(domain=FREQ_ORDER, range=FREQ_COLORS),
            legend=alt.Legend(title=None),
        ),
        order=alt.Order("Order:Q"),
        tooltip=[
            "Frequency:N",
            "Respondents:Q",
            alt.Tooltip("Percent:Q", format=".1f", title="Percent"),
        ],
    )
)

frequency_text = (
    alt.Chart(frequency_summary.loc[frequency_summary["Percent"].ge(7)])
    .mark_text(color="white", fontWeight="bold", fontSize=12)
    .encode(
        x="Midpoint:Q",
        y=alt.Y("Metric:N"),
        text="Label:N",
    )
)

frequency_chart = style_chart(
    (frequency_bar + frequency_text).properties(
        width=820,
        height=70,
        title=alt.Title(
            f"{ever_used:.0f}% used generative AI; {weekly_used:.0f}% used it weekly or daily",
            subtitle=f"2025 cross-section; valid AI-frequency responses n={len(ai_respondents):,}",
        ),
    )
)
frequency_chart

alt.LayerChart(...)

In [6]:
# %% [markdown]
# ## Frequency of generative AI use: proportional waffle chart

# %%
frequency_waffle_summary = (
    ai_respondents[FREQUENCY]
    .value_counts()
    .reindex(FREQ_ORDER, fill_value=0)
    .rename_axis("Frequency")
    .reset_index(name="Respondents")
)

frequency_waffle_summary["Percent"] = (
    100
    * frequency_waffle_summary["Respondents"]
    / frequency_waffle_summary["Respondents"].sum()
)

# Use 200 equal-area tiles: one tile = 0.5 percentage points.
N_COLS = 20
N_ROWS = 10
N_TILES = N_COLS * N_ROWS

# Largest-remainder allocation ensures the chart contains exactly 200 tiles.
raw_tiles = frequency_waffle_summary["Percent"] * N_TILES / 100
whole_tiles = np.floor(raw_tiles).astype(int)

frequency_waffle_summary["Tiles"] = whole_tiles

tiles_remaining = int(
    N_TILES - frequency_waffle_summary["Tiles"].sum()
)

if tiles_remaining:
    remainder_order = (
        raw_tiles - whole_tiles
    ).nlargest(tiles_remaining).index

    frequency_waffle_summary.loc[
        remainder_order, "Tiles"
    ] += 1

frequency_waffle_summary["LegendLabel"] = (
    frequency_waffle_summary["Frequency"]
    + " · "
    + frequency_waffle_summary["Percent"].map(lambda x: f"{x:.1f}%")
)

# Expand each response category into its allocated number of tiles.
waffle_tiles = pd.DataFrame(
    {
        "Frequency": np.repeat(
            frequency_waffle_summary["Frequency"].to_numpy(),
            frequency_waffle_summary["Tiles"].to_numpy(),
        )
    }
)

waffle_tiles["Tile"] = np.arange(N_TILES)
waffle_tiles["Column"] = waffle_tiles["Tile"] % N_COLS

# Fill from the lower-left corner, reading left to right.
waffle_tiles["Row"] = (
    N_ROWS - 1 - waffle_tiles["Tile"] // N_COLS
)

waffle_tiles = waffle_tiles.merge(
    frequency_waffle_summary[
        [
            "Frequency",
            "Respondents",
            "Percent",
            "Tiles",
            "LegendLabel",
        ]
    ],
    on="Frequency",
    how="left",
)

legend_order = frequency_waffle_summary["LegendLabel"].tolist()

# Light-to-dark progression while remaining within the CU visual family.
WAFFLE_COLORS = [
    "#E4E4E4",  # Never
    "#D8C99B",  # Several times per year
    "#CFB87C",  # Several times per month
    "#565A5C",  # Several times per week
    "#000000",  # Daily
]

frequency_waffle_chart = style_chart(
    alt.Chart(waffle_tiles)
    .mark_rect(
        stroke="white",
        strokeWidth=1.25,
        cornerRadius=1,
    )
    .encode(
        x=alt.X(
            "Column:O",
            axis=None,
            scale=alt.Scale(
                paddingInner=0.06,
                paddingOuter=0,
            ),
        ),
        y=alt.Y(
            "Row:O",
            axis=None,
            scale=alt.Scale(
                paddingInner=0.06,
                paddingOuter=0,
            ),
        ),
        color=alt.Color(
            "LegendLabel:N",
            sort=legend_order,
            scale=alt.Scale(
                domain=legend_order,
                range=WAFFLE_COLORS,
            ),
            legend=alt.Legend(
                title=None,
                orient="top",
                columns=3,
                symbolType="square",
                symbolSize=120,
                labelLimit=300,
            ),
        ),
        tooltip=[
            alt.Tooltip("Frequency:N", title="Frequency"),
            alt.Tooltip(
                "Respondents:Q",
                title="Respondents",
                format=",",
            ),
            alt.Tooltip(
                "Percent:Q",
                title="Exact share",
                format=".1f",
            ),
        ],
    )
    .properties(
        width=760,
        height=380,
        title=alt.Title(
            "How often graduate students use generative AI",
            subtitle=(
                f"Valid responses n={len(ai_respondents):,}; "
                "200 equal-area tiles, each representing 0.5 percentage points"
            ),
        ),
    )
)

frequency_waffle_chart

alt.Chart(...)

In [7]:
# Purpose prevalence among respondents with a complete, eligible multiselect battery.
purpose_summary = (
    purpose_long.groupby(["code", "Purpose"], observed=True)
    .agg(Selected=("Selected", "sum"), Base=("Selected", "size"))
    .reset_index()
)
purpose_summary["Percent"] = 100 * purpose_summary["Selected"] / purpose_summary["Base"]
purpose_order = purpose_summary.sort_values("Percent")["Purpose"].tolist()

purpose_stems = (
    alt.Chart(purpose_summary)
    .mark_rule(color="#C8C8C8", strokeWidth=2)
    .encode(
        y=alt.Y("Purpose:N", sort=purpose_order, title=None),
        x=alt.X("x0:Q", title="% selecting purpose", scale=alt.Scale(domain=[0, 65])),
        x2="Percent:Q",
    )
    .transform_calculate(x0="0")
)
purpose_points = (
    alt.Chart(purpose_summary)
    .mark_circle(size=115, color=CU_GOLD, stroke=CU_BLACK, strokeWidth=0.5)
    .encode(
        y=alt.Y("Purpose:N", sort=purpose_order),
        x="Percent:Q",
        tooltip=[
            "Purpose:N",
            "Selected:Q",
            "Base:Q",
            alt.Tooltip("Percent:Q", format=".1f", title="Percent"),
        ],
    )
)
purpose_labels = (
    alt.Chart(purpose_summary)
    .mark_text(align="left", dx=8, fontSize=11)
    .encode(
        y=alt.Y("Purpose:N", sort=purpose_order),
        x="Percent:Q",
        text=alt.Text("Percent:Q", format=".0f"),
    )
)

purpose_chart = style_chart(
    (purpose_stems + purpose_points + purpose_labels).properties(
        width=650,
        height=315,
        title=alt.Title(
            "Research, coding, and brainstorming dominate reported uses",
            subtitle=f"Share of AI users with a complete purpose battery; n={len(purpose_base):,}; multiselect",
        ),
    )
)
purpose_chart

alt.LayerChart(...)

In [16]:
LIKERT_COLORS_REVISED = [CU_BLACK, CU_DARK_GRAY, "#E6D6A9", CU_GOLD]

attitude_distribution = (
    attitude_long.groupby(["code", "Item", "Response"], observed=True)
    .size()
    .rename("Respondents")
    .reset_index()
)

LIKERT_STACK_ORDER = {
    "Disagree": 0,          
    "Strongly disagree": 1, 
    "Agree": 2,             
    "Strongly agree": 3,    
}

attitude_distribution["ItemBase"] = attitude_distribution.groupby("code")[
    "Respondents"
].transform("sum")
attitude_distribution["Percent"] = (
    100 * attitude_distribution["Respondents"] / attitude_distribution["ItemBase"]
)
attitude_distribution["SignedPercent"] = np.where(
    attitude_distribution["Response"].isin(["Strongly disagree", "Disagree"]),
    -attitude_distribution["Percent"],
    attitude_distribution["Percent"],
)
attitude_distribution["StackOrder"] = attitude_distribution["Response"].map(LIKERT_STACK_ORDER)


attitude_topbox = (
    attitude_long.assign(
        Agree=lambda frame: frame["Response"].isin(["Agree", "Strongly agree"])
    )
    .groupby(["code", "Item"], observed=True)["Agree"]
    .agg(["mean", "size"])
    .reset_index()
    .rename(columns={"mean": "TopBox", "size": "ItemBase"})
)
attitude_topbox["ItemLabel"] = attitude_topbox.apply(
    lambda row: f"{row['Item']}|(n={row['ItemBase']:,})",
    axis=1,
)
attitude_item_labels = attitude_topbox.set_index("code")["ItemLabel"].to_dict()
attitude_distribution["ItemLabel"] = attitude_distribution["code"].map(
    attitude_item_labels
)
attitude_order = (
    attitude_topbox.sort_values("TopBox")["code"]
    .map(attitude_item_labels)
    .tolist()
)

likert_bars_bordered = (
    alt.Chart(attitude_distribution)
    .mark_bar(stroke="#9C9C9C", strokeWidth=1, fillOpacity=.9, size=50)
    .encode(
        y=alt.Y(
            "ItemLabel:N",
            sort=attitude_order,
            title=None,
            axis=alt.Axis(
                labelExpr="split(datum.label, '|')",

                labelLineHeight=16,
                labelLimit=500,
                labelPadding=12,
            ),
),
        x=alt.X(
            "SignedPercent:Q",
            stack="zero",
            title="Share of valid item responses",
            axis=alt.Axis(labelExpr="abs(datum.value) + '%'"),
            scale=alt.Scale(domain=[-65, 100]),
        ),
        color=alt.Color(
            "Response:N",
            sort=LIKERT_ORDER,
            scale=alt.Scale(
                domain=LIKERT_ORDER,
                range=LIKERT_COLORS_REVISED,
            ),
            legend=alt.Legend(title=None),
        ),
        order=alt.Order("StackOrder:Q"),
        tooltip=[
            "Item:N",
            "Response:N",
            "Respondents:Q",
            "ItemBase:Q",
            alt.Tooltip("Percent:Q", format=".1f", title="Percent"),
        ],
    )
)

attitude_zero = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color=CU_BLACK, strokeWidth=1.5)
    .encode(x="x:Q")
)

attitude_chart_bordered = style_chart(
    (likert_bars_bordered + attitude_zero).properties(
        width=1200,
        height=500,
        title=alt.Title(
            "Students report more understanding than institutional guidance",
            subtitle="Item-specific valid bases; teaching and research/writing items were conditionally displayed",
        ),
    )
)



attitude_chart_bordered

alt.LayerChart(...)

### An alluvial view of the guidance gap

This is a true respondent-conserving flow: each person appears once, and all three stages are
mutually exclusive. It avoids the duplicated-person problem that would arise from putting the
multiselect purpose battery directly into a Sankey.

In [9]:
def alluvial_layout(data, stages, stage_orders, weight_name="Respondents", gap=28, steps=24):
    # Return ribbon and node coordinates for a respondent-conserving alluvial chart.
    paths = (
        data.groupby(stages, observed=True)
        .size()
        .rename(weight_name)
        .reset_index()
    )
    paths["path_id"] = np.arange(len(paths))

    rank = {
        stage: {category: index for index, category in enumerate(stage_orders[stage])}
        for stage in stages
    }
    positions = {}
    node_rows = []

    for stage_index, stage in enumerate(stages):
        totals = paths.groupby(stage, observed=True)[weight_name].sum()
        cursor = 0.0
        for category in stage_orders[stage]:
            if category not in totals:
                continue
            node_total = float(totals[category])
            node_start = cursor
            subset = paths.loc[paths[stage].eq(category)].copy()
            other_stages = [column for column in stages if column != stage]
            subset["_sort"] = subset.apply(
                lambda row: tuple(rank[column][row[column]] for column in other_stages),
                axis=1,
            )
            subset = subset.sort_values("_sort")

            path_cursor = node_start
            for row in subset.itertuples(index=False):
                path_id = row.path_id
                weight = float(getattr(row, weight_name))
                positions[(stage_index, path_id)] = (path_cursor, path_cursor + weight)
                path_cursor += weight

            node_rows.append(
                {
                    "Stage": stage,
                    "StageIndex": stage_index,
                    "Category": category,
                    "Respondents": node_total,
                    "x0": stage_index - 0.035,
                    "x1": stage_index + 0.035,
                    "y0": node_start,
                    "y1": node_start + node_total,
                    "ymid": node_start + node_total / 2,
                }
            )
            cursor += node_total + gap

    ribbon_rows = []
    for _, row in paths.iterrows():
        path_id = int(row["path_id"])
        path_values = {stage: row[stage] for stage in stages}
        for segment in range(len(stages) - 1):
            left_low, left_high = positions[(segment, path_id)]
            right_low, right_high = positions[(segment + 1, path_id)]
            for fraction in np.linspace(0, 1, steps):
                smooth = fraction * fraction * (3 - 2 * fraction)
                ribbon_rows.append(
                    {
                        "Ribbon": f"{path_id}-{segment}",
                        "x": segment + fraction,
                        "y0": left_low + smooth * (right_low - left_low),
                        "y1": left_high + smooth * (right_high - left_high),
                        "Respondents": row[weight_name],
                        **path_values,
                    }
                )

    return pd.DataFrame(ribbon_rows), pd.DataFrame(node_rows)


flow_stages = ["usage_tier", "Faculty discussion", "Appropriate-use understanding"]
flow_orders = {
    "usage_tier": USAGE_TIER_ORDER,
    "Faculty discussion": ["Not discussed", "Discussed"],
    "Appropriate-use understanding": ["Does not understand", "Understands"],
}
flow_ribbons, flow_nodes = alluvial_layout(guidance_paths, flow_stages, flow_orders)

ribbons = (
    alt.Chart(flow_ribbons)
    .mark_area(opacity=0.48)
    .encode(
        x=alt.X("x:Q", axis=None, scale=alt.Scale(domain=[-0.35, 2.35])),
        y=alt.Y("y0:Q", axis=None),
        y2="y1:Q",
        detail="Ribbon:N",
        color=alt.Color(
            "usage_tier:N",
            sort=USAGE_TIER_ORDER,
            scale=alt.Scale(
                domain=USAGE_TIER_ORDER,
                range=[CU_LIGHT_GRAY, CU_GOLD, CU_BLACK],
            ),
            legend=alt.Legend(title="AI-use intensity"),
        ),
        tooltip=[
            alt.Tooltip("usage_tier:N", title="AI-use intensity"),
            "Faculty discussion:N",
            "Appropriate-use understanding:N",
            "Respondents:Q",
        ],
    )
)
nodes = (
    alt.Chart(flow_nodes)
    .mark_rect(color=CU_DARK_GRAY)
    .encode(x="x0:Q", x2="x1:Q", y="y0:Q", y2="y1:Q")
)

left_labels = (
    alt.Chart(flow_nodes.loc[flow_nodes["StageIndex"].eq(0)])
    .mark_text(align="right", dx=-8, fontSize=11)
    .encode(x="x0:Q", y="ymid:Q", text="Category:N")
)
middle_labels = (
    alt.Chart(flow_nodes.loc[flow_nodes["StageIndex"].eq(1)])
    .mark_text(align="center", baseline="middle", color="white", fontSize=10)
    .encode(x="StageIndex:Q", y="ymid:Q", text="Category:N")
)
right_labels = (
    alt.Chart(flow_nodes.loc[flow_nodes["StageIndex"].eq(2)])
    .mark_text(align="left", dx=8, fontSize=11)
    .encode(x="x1:Q", y="ymid:Q", text="Category:N")
)

alluvial_chart = style_chart(
    (ribbons + nodes + left_labels + middle_labels + right_labels).properties(
        width=860,
        height=480,
        title=alt.Title(
            "AI-use intensity → faculty discussion → appropriate-use understanding",
            subtitle=f"Flow Shows which portions of each use frequency ={len(guidance_paths):,}",
        ),
    )
)
alluvial_chart

alt.LayerChart(...)

In [10]:
def threaded_alluvial_layout(
    data,
    stages,
    stage_orders,
    weight_name="Respondents",
    gap=28,
    steps=25,
):
    paths = (
        data.groupby(stages, observed=True)
        .size()
        .rename(weight_name)
        .reset_index()
    )
    paths["path_id"] = np.arange(len(paths))
    paths["Thread"] = paths[stages].astype(str).agg(" → ".join, axis=1)

    rank = {
        stage: {category: index for index, category in enumerate(stage_orders[stage])}
        for stage in stages
    }
    positions = {}
    node_rows = []

    for stage_index, stage in enumerate(stages):
        totals = paths.groupby(stage, observed=True)[weight_name].sum()
        node_cursor = 0.0

        for category in stage_orders[stage]:
            if category not in totals:
                continue

            node_total = float(totals[category])
            subset = paths.loc[paths[stage].eq(category)].copy()

            # Favor clean separation for the discussion → understanding transition.
            if stage_index == 0:
                sort_columns = [stages[1], stages[2]]
            elif stage_index == 1:
                sort_columns = [stages[0], stages[2]]
            else:
                sort_columns = [stages[1], stages[0]]

            subset["_sort"] = subset.apply(
                lambda row: tuple(rank[column][row[column]] for column in sort_columns),
                axis=1,
            )
            subset = subset.sort_values("_sort")

            path_cursor = node_cursor
            for _, row in subset.iterrows():
                path_id = int(row["path_id"])
                weight = float(row[weight_name])
                positions[(stage_index, path_id)] = (
                    path_cursor,
                    path_cursor + weight,
                )
                path_cursor += weight

            node_rows.append(
                {
                    "Stage": stage,
                    "StageIndex": stage_index,
                    "Category": category,
                    "Respondents": node_total,
                    "x0": stage_index - 0.035,
                    "x1": stage_index + 0.035,
                    "y0": node_cursor,
                    "y1": node_cursor + node_total,
                    "ymid": node_cursor + node_total / 2,
                }
            )
            node_cursor += node_total + gap

    ribbon_rows = []
    for _, row in paths.iterrows():
        path_id = int(row["path_id"])
        path_values = {stage: row[stage] for stage in stages}

        for segment in range(len(stages) - 1):
            left_low, left_high = positions[(segment, path_id)]
            right_low, right_high = positions[(segment + 1, path_id)]

            for fraction in np.linspace(0, 1, steps):
                smooth = fraction * fraction * (3 - 2 * fraction)
                ribbon_rows.append(
                    {
                        "Ribbon": f"{path_id}-{segment}",
                        "Thread": row["Thread"],
                        "Segment": segment,
                        "x": segment + fraction,
                        "y0": left_low + smooth * (right_low - left_low),
                        "y1": left_high + smooth * (right_high - left_high),
                        "Respondents": row[weight_name],
                        **path_values,
                    }
                )

    nodes = pd.DataFrame(node_rows)
    nodes["Percent"] = nodes["Respondents"] / len(data)
    nodes["NodeLabel"] = nodes.apply(
        lambda row: f"{row['Category']} · {row['Percent']:.0%}",
        axis=1,
    )

    return pd.DataFrame(ribbon_rows), nodes, paths


thread_stages = [
    "Faculty discussion",
    "Appropriate-use understanding",
    "usage_tier",
]
thread_orders = {
    "Faculty discussion": ["Not discussed", "Discussed"],
    "Appropriate-use understanding": ["Does not understand", "Understands"],
    "usage_tier": USAGE_TIER_ORDER,
}

thread_ribbons, thread_nodes, thread_paths = threaded_alluvial_layout(
    guidance_paths,
    thread_stages,
    thread_orders,
)

# Twelve related but separable colors: gray family = not discussed;
# gold family = discussed; tone varies by understanding and use intensity.
THREAD_COLOR = {
    "Not discussed → Does not understand → Never": "#D9DBDC",
    "Not discussed → Does not understand → Occasional": "#6D7172",
    "Not discussed → Does not understand → Frequent": "#101112",
    "Not discussed → Understands → Never": "#BEC1C2",
    "Not discussed → Understands → Occasional": "#8C9091",
    "Not discussed → Understands → Frequent": "#3E4142",
    "Discussed → Does not understand → Never": "#D5C28E",
    "Discussed → Does not understand → Occasional": "#B89E60",
    "Discussed → Does not understand → Frequent": "#665B3F",
    "Discussed → Understands → Never": "#E7DDBF",
    "Discussed → Understands → Occasional": CU_GOLD,
    "Discussed → Understands → Frequent": "#8A7950",
}

thread_domain = list(THREAD_COLOR)
thread_range = [THREAD_COLOR[thread] for thread in thread_domain]

thread_areas = (
    alt.Chart(thread_ribbons)
    .mark_area(opacity=0.9, stroke="white", strokeWidth=0.8)
    .encode(
        x=alt.X(
            "x:Q",
            axis=None,
            scale=alt.Scale(domain=[-0.45, 2.48]),
        ),
        y=alt.Y("y0:Q", axis=None),
        y2="y1:Q",
        detail="Ribbon:N",
        color=alt.Color(
            "Thread:N",
            scale=alt.Scale(domain=thread_domain, range=thread_range),
            legend=None,
        ),
        tooltip=[
            "Faculty discussion:N",
            "Appropriate-use understanding:N",
            alt.Tooltip("usage_tier:N", title="AI-use intensity"),
            "Respondents:Q",
        ],
    )
)

thread_node_marks = (
    alt.Chart(thread_nodes)
    .mark_rect(color=CU_BLACK, stroke="white", strokeWidth=0.7)
    .encode(
        x="x0:Q",
        x2="x1:Q",
        y="y0:Q",
        y2="y1:Q",
        tooltip=[
            "Stage:N",
            "Category:N",
            "Respondents:Q",
            alt.Tooltip("Percent:Q", format=".1%", title="Share of base"),
        ],
    )
)

# Place every node label adjacent to its own stage bar.
thread_nodes["LabelX"] = np.where(
    thread_nodes["StageIndex"].eq(0),
    thread_nodes["x0"] - 0.055,
    thread_nodes["x1"] + 0.055,
)
thread_nodes["LabelAlign"] = np.where(
    thread_nodes["StageIndex"].eq(0),
    "right",
    "left",
)

left_node_labels = thread_nodes.loc[thread_nodes["StageIndex"].eq(0)]
right_node_labels = thread_nodes.loc[thread_nodes["StageIndex"].ne(0)]

left_node_halo = (
    alt.Chart(left_node_labels)
    .mark_text(
        align="right",
        fontSize=11,
        fontWeight="bold",
        stroke="white",
        strokeWidth=4,
    )
    .encode(x="LabelX:Q", y="ymid:Q", text="NodeLabel:N")
)
left_node_text = (
    alt.Chart(left_node_labels)
    .mark_text(align="right", fontSize=11, fontWeight="bold", color=CU_BLACK)
    .encode(x="LabelX:Q", y="ymid:Q", text="NodeLabel:N")
)
right_node_halo = (
    alt.Chart(right_node_labels)
    .mark_text(
        align="left",
        fontSize=11,
        fontWeight="bold",
        stroke="white",
        strokeWidth=4,
    )
    .encode(x="LabelX:Q", y="ymid:Q", text="NodeLabel:N")
)
right_node_text = (
    alt.Chart(right_node_labels)
    .mark_text(align="left", fontSize=11, fontWeight="bold", color=CU_BLACK)
    .encode(x="LabelX:Q", y="ymid:Q", text="NodeLabel:N")
)

# Two conditional percentages make discussion ↔ understanding legible.
discussion_understanding = (
    thread_paths.groupby(
        ["Faculty discussion", "Appropriate-use understanding"],
        observed=True,
    )["Respondents"]
    .sum()
    .rename("TransitionN")
    .reset_index()
)
discussion_understanding["SourceN"] = discussion_understanding.groupby(
    "Faculty discussion"
)["TransitionN"].transform("sum")
discussion_understanding["ConditionalPercent"] = (
    discussion_understanding["TransitionN"]
    / discussion_understanding["SourceN"]
)

transition_labels = discussion_understanding.loc[
    discussion_understanding["Appropriate-use understanding"].eq("Understands")
].merge(
    thread_nodes.loc[
        thread_nodes["Stage"].eq("Faculty discussion"),
        ["Category", "ymid"],
    ],
    left_on="Faculty discussion",
    right_on="Category",
)
transition_labels["x"] = 0.5
transition_labels["Label"] = transition_labels["ConditionalPercent"].map(
    lambda value: f"{value:.0%} understand"
)

transition_halo = (
    alt.Chart(transition_labels)
    .mark_text(
        fontSize=10,
        fontWeight="bold",
        stroke="white",
        strokeWidth=4,
    )
    .encode(
        x="x:Q",
        y="ymid:Q",
        text="Label:N",
        tooltip=[
            "Faculty discussion:N",
            "Appropriate-use understanding:N",
            "TransitionN:Q",
            alt.Tooltip(
                "ConditionalPercent:Q",
                format=".1%",
                title="Within discussion status",
            ),
        ],
    )
)
transition_text = (
    alt.Chart(transition_labels)
    .mark_text(fontSize=10, fontWeight="bold", color=CU_BLACK)
    .encode(x="x:Q", y="ymid:Q", text="Label:N")
)

header_y = thread_nodes["y1"].max() + 55
stage_headers = pd.DataFrame(
    {
        "x": [0, 1, 2],
        "y": [header_y] * 3,
        "Header": [
            "1  FACULTY DISCUSSION",
            "2  APPROPRIATE-USE UNDERSTANDING",
            "3  AI-USE INTENSITY",
        ],
    }
)
stage_header_text = (
    alt.Chart(stage_headers)
    .mark_text(
        align="center",
        fontSize=12,
        fontWeight="bold",
        color=CU_BLACK,
    )
    .encode(x="x:Q", y="y:Q", text="Header:N")
)

sankey_threaded_chart = style_chart(
    (
        thread_areas
        + thread_node_marks
        + left_node_halo
        + left_node_text
        + right_node_halo
        + right_node_text
        + transition_halo
        + transition_text
        + stage_header_text
    ).properties(
        width=900,
        height=520,
        title=alt.Title(
            "Discussion, understanding, and AI-use intensity",
            subtitle=(
                f"One conserved thread per joint response pattern; n={len(guidance_paths):,}. "
                "Layout is associational, not causal."
            ),
        ),
    )
)

sankey_threaded_chart


alt.LayerChart(...)

In [11]:
# College-by-purpose grid. Display rates and color the percentage-point difference from
# the institution-wide purpose rate. Suppress college bases below 30.
college_bases = purpose_base.groupby("COLLEGE_NAME1").size()
eligible_colleges = college_bases.loc[college_bases.ge(30)].index

college_purpose = (
    purpose_long.loc[purpose_long["COLLEGE_NAME1"].isin(eligible_colleges)]
    .groupby(["COLLEGE_NAME1", "code", "PurposeShort"], observed=True)
    .agg(Selected=("Selected", "sum"), Base=("Selected", "size"))
    .reset_index()
)
college_purpose["Percent"] = 100 * college_purpose["Selected"] / college_purpose["Base"]

institution_purpose = purpose_long.groupby("code", observed=True)["Selected"].mean().mul(100)
college_purpose["Overall"] = college_purpose["code"].map(institution_purpose)
college_purpose["Delta"] = college_purpose["Percent"] - college_purpose["Overall"]
college_purpose["RateLabel"] = college_purpose["Percent"].map(lambda value: f"{value:.0f}%")
college_purpose["College"] = college_purpose["COLLEGE_NAME1"].map(COLLEGE_SHORT)
college_purpose["CollegeLabel"] = college_purpose.apply(
    lambda row: f"{row['College']}  (n={row['Base']:,})", axis=1
)

purpose_short_order = [PURPOSE_SHORT[code] for code in PURPOSE_COLUMNS]
college_order = (
    college_bases.loc[eligible_colleges]
    .sort_values(ascending=False)
    .index.to_series()
    .map(COLLEGE_SHORT)
    .tolist()
)
college_label_order = [
    f"{college}  (n={int(college_bases.loc[name]):,})"
    for name, college in zip(
        college_bases.loc[eligible_colleges].sort_values(ascending=False).index,
        college_order,
    )
]

heatmap = (
    alt.Chart(college_purpose)
    .mark_rect(cornerRadius=2)
    .encode(
        x=alt.X(
            "PurposeShort:N",
            sort=purpose_short_order,
            title=None,
            axis=alt.Axis(labelAngle=-38, labelAlign="right"),
        ),
        y=alt.Y("CollegeLabel:N", sort=college_label_order, title=None),
        color=alt.Color(
            "Delta:Q",
            scale=alt.Scale(
                type="symlog",
                domain=[-40, 25],
                domainMid=0,
                range=[CU_BLACK,  "#E4E4E4",  CU_GOLD],
            ),
            legend=alt.Legend(title="Difference from overall (pp)"),
        ),
        tooltip=[
            alt.Tooltip("College:N"),
            alt.Tooltip("PurposeShort:N", title="Purpose"),
            "Selected:Q",
            "Base:Q",
            alt.Tooltip("Percent:Q", format=".1f", title="College percent"),
            alt.Tooltip("Overall:Q", format=".1f", title="Overall percent"),
            alt.Tooltip("Delta:Q", format="+.1f", title="Difference (pp)"),
        ],
    )
)
heatmap_text = (
    alt.Chart(college_purpose)
    .mark_text(fontSize=10, fontWeight="bold")
    .encode(
        x=alt.X("PurposeShort:N", sort=purpose_short_order),
        y=alt.Y("CollegeLabel:N", sort=college_label_order),
        text="RateLabel:N",
        color=alt.condition(
            "abs(datum.Delta) >= 5",
            alt.value("white"),
            alt.value(CU_BLACK),
        ),
    )
)

college_purpose_chart = style_chart(
    (heatmap + heatmap_text).properties(
        width=1080,
        height=500,
        title=alt.Title(
            "Purpose profiles reveal discipline-specific patterns",
            subtitle="Cell text = within-college rate; color = percentage-point difference from all complete purpose batteries; college n≥30",
        ),
    )
)
college_purpose_chart

alt.LayerChart(...)

## 4. Diagnostic appendix

The following checks support interview questions but do not need to appear in the main deck.
They quantify uncertainty in small college samples and test whether the college pattern is
primarily a composition effect from `LEVEL_GRAD`.

In [12]:
# Weekly-or-more use by college with Wilson 95% intervals and explicit sample-size signaling.
college_weekly = (
    ai_respondents.groupby(["COLLEGE_CODE1", "COLLEGE_NAME1"], observed=True)
    .agg(
        Respondents=(FREQUENCY, "size"),
        Weekly=("weekly_or_more", "sum"),
        Percent=("weekly_or_more", "mean"),
    )
    .reset_index()
)
college_weekly["Percent"] *= 100

z = 1.96
p = college_weekly["Weekly"] / college_weekly["Respondents"]
denominator = 1 + z**2 / college_weekly["Respondents"]
center = (p + z**2 / (2 * college_weekly["Respondents"])) / denominator
half_width = (
    z
    * np.sqrt(
        p * (1 - p) / college_weekly["Respondents"]
        + z**2 / (4 * college_weekly["Respondents"] ** 2)
    )
    / denominator
)
college_weekly["Lower"] = 100 * (center - half_width)
college_weekly["Upper"] = 100 * (center + half_width)
college_weekly["College"] = college_weekly["COLLEGE_NAME1"].map(COLLEGE_SHORT)
college_weekly["CollegeLabel"] = college_weekly.apply(
    lambda row: f"{row['College']}  (n={row['Respondents']:,})", axis=1
)
college_weekly["SampleFlag"] = np.where(
    college_weekly["Respondents"].ge(30), "n ≥ 30", "n < 30"
)
college_sort = college_weekly.sort_values("Percent")["CollegeLabel"].tolist()
institution_weekly = 100 * ai_respondents["weekly_or_more"].mean()

intervals = (
    alt.Chart(college_weekly)
    .mark_rule(strokeWidth=2)
    .encode(
        y=alt.Y("CollegeLabel:N", sort=college_sort, title=None),
        x=alt.X("Lower:Q", title="% using AI weekly or daily", scale=alt.Scale(domain=[0, 105])),
        x2="Upper:Q",
        color=alt.Color(
            "SampleFlag:N",
            scale=alt.Scale(
                domain=["n ≥ 30", "n < 30"],
                range=[CU_GOLD, "#A9ADB0"],
            ),
            legend=alt.Legend(title=None),
        ),
    )
)
points = (
    alt.Chart(college_weekly)
    .mark_circle(size=120, stroke=CU_BLACK, strokeWidth=0.5)
    .encode(
        y=alt.Y("CollegeLabel:N", sort=college_sort),
        x="Percent:Q",
        color=alt.Color(
            "SampleFlag:N",
            scale=alt.Scale(domain=["n ≥ 30", "n < 30"], range=[CU_GOLD, "#A9ADB0"]),
            legend=None,
        ),
        tooltip=[
            "College:N",
            "Respondents:Q",
            "Weekly:Q",
            alt.Tooltip("Percent:Q", format=".1f"),
            alt.Tooltip("Lower:Q", format=".1f"),
            alt.Tooltip("Upper:Q", format=".1f"),
        ],
    )
)
institution_rule = (
    alt.Chart(pd.DataFrame({"Percent": [institution_weekly]}))
    .mark_rule(color=CU_BLACK, strokeDash=[5, 4])
    .encode(x="Percent:Q")
)

college_chart = style_chart(
    (intervals + points + institution_rule).properties(
        width=720,
        height=330,
        title=alt.Title(
            "College estimates vary, but small groups are highly uncertain",
            subtitle=f"Wilson 95% intervals; dashed line = institution respondent rate ({institution_weekly:.1f}%)",
        ),
    )
)
college_chart

alt.LayerChart(...)

In [13]:
# Back-pocket composition check: categorical adjustment only; no unsupported level labels.
model_data = ai_respondents.dropna(
    subset=["COLLEGE_CODE1", "COLLEGE_NAME1", "LEVEL_GRAD"]
).copy()
model_data["weekly"] = model_data["weekly_or_more"].astype(int)
model_data["level_code"] = model_data["LEVEL_GRAD"].astype(int).astype(str)

weekly_model = smf.logit(
    "weekly ~ C(COLLEGE_CODE1) + C(level_code)",
    data=model_data,
).fit(disp=False)

adjusted_rows = []
for college_code, college_name in (
    model_data[["COLLEGE_CODE1", "COLLEGE_NAME1"]]
    .drop_duplicates()
    .sort_values("COLLEGE_CODE1")
    .itertuples(index=False)
):
    scenario = model_data.copy()
    scenario["COLLEGE_CODE1"] = college_code
    college_observed = model_data.loc[model_data["COLLEGE_CODE1"].eq(college_code)]
    adjusted_rows.append(
        {
            "College": COLLEGE_SHORT[college_name],
            "n": len(college_observed),
            "Observed": 100 * college_observed["weekly"].mean(),
            "Adjusted": 100 * weekly_model.predict(scenario).mean(),
        }
    )

college_adjustment = pd.DataFrame(adjusted_rows)
college_adjustment["Change"] = college_adjustment["Adjusted"] - college_adjustment["Observed"]
adjustment_sort = college_adjustment.sort_values("Observed")["College"].tolist()

adjustment_lines = (
    alt.Chart(college_adjustment)
    .mark_rule(color="#A9ADB0", strokeWidth=2)
    .encode(
        y=alt.Y("College:N", sort=adjustment_sort, title=None),
        x=alt.X("Observed:Q", title="% using AI weekly or daily", scale=alt.Scale(domain=[0, 100])),
        x2="Adjusted:Q",
    )
)
adjustment_points = (
    alt.Chart(
        college_adjustment.melt(
            id_vars=["College", "n"],
            value_vars=["Observed", "Adjusted"],
            var_name="Estimate",
            value_name="Percent",
        )
    )
    .mark_point(filled=True, size=105)
    .encode(
        y=alt.Y("College:N", sort=adjustment_sort, title=None),
        x="Percent:Q",
        color=alt.Color(
            "Estimate:N",
            scale=alt.Scale(
                domain=["Observed", "Adjusted"],
                range=[CU_DARK_GRAY, CU_GOLD],
            ),
            legend=alt.Legend(title=None),
        ),
        tooltip=[
            "College:N",
            "n:Q",
            "Estimate:N",
            alt.Tooltip("Percent:Q", format=".1f"),
        ],
    )
)

adjustment_chart = style_chart(
    (adjustment_lines + adjustment_points).properties(
        width=700,
        height=310,
        title=alt.Title(
            "Degree-level composition changes college estimates very little",
            subtitle="Logistic standardization using LEVEL_GRAD as an unlabeled categorical code; descriptive check, not a causal model",
        ),
    )
)

display(
    college_adjustment.sort_values("Observed", ascending=False)
    .style.hide(axis="index")
    .format({"n": "{:,.0f}", "Observed": "{:.1f}%", "Adjusted": "{:.1f}%", "Change": "{:+.1f} pp"})
    .set_caption(
        f"Model converged: {weekly_model.mle_retvals['converged']}; "
        f"maximum absolute adjustment = {college_adjustment['Change'].abs().max():.1f} percentage points"
    )
)
adjustment_chart

College,n,Observed,Adjusted,Change
Business,14,92.9%,92.3%,-0.6 pp
Engineering,393,34.6%,34.6%,-0.0 pp
Arts & Sciences,611,25.5%,25.3%,-0.3 pp
"Media, Comm & Info",55,23.6%,23.6%,-0.0 pp
Education,51,19.6%,21.2%,+1.6 pp
Cross-College,11,9.1%,11.0%,+1.9 pp
Music,42,4.8%,5.1%,+0.3 pp


alt.LayerChart(...)

In [14]:
# Metadata audit: retain codes until an authoritative crosswalk is supplied.
level_audit = (
    survey.groupby("LEVEL_GRAD", observed=True)
    .agg(
        Respondents=("respondent_id", "size"),
        Programs=("PROGRAM_CODE1", "nunique"),
        EarliestEntry=("ENTRY_YEAR", "min"),
        LatestEntry=("ENTRY_YEAR", "max"),
    )
    .reset_index()
    .rename(columns={"LEVEL_GRAD": "LEVEL_GRAD code"})
)

entry_years = pd.to_numeric(survey["ENTRY_YEAR"], errors="coerce")
entry_note = pd.DataFrame(
    {
        "Field": ["ENTRY_YEAR / ENTRY_TERM", "LEVEL_GRAD"],
        "Observed evidence": [
            (
                f"Entry years span {entry_years.min():.0f}–{entry_years.max():.0f}; "
                f"terms include {', '.join(sorted(survey['ENTRY_TERM'].dropna().unique()))}"
            ),
            "Observed codes: "
            + ", ".join(map(str, sorted(survey["LEVEL_GRAD"].dropna().astype(int).unique()))),
        ],
        "Analytical treatment": [
            "Entry cohort metadata; never interpreted as survey year",
            "Categorical code only; no unsupported degree labels",
        ],
    }
)

display(
    level_audit.style.hide(axis="index")
    .format(
        {
            "LEVEL_GRAD code": "{:.0f}",
            "Respondents": "{:,.0f}",
            "Programs": "{:,.0f}",
            "EarliestEntry": "{:.0f}",
            "LatestEntry": "{:.0f}",
        }
    )
    .set_caption("Unlabeled degree-level code audit")
)
display(entry_note.style.hide(axis="index").set_caption("Metadata interpretation decisions"))

LEVEL_GRAD code,Respondents,Programs,EarliestEntry,LatestEntry
1,327,45,2006,2025
3,"1,054",56,2005,2025
4,6,1,2021,2024


Field,Observed evidence,Analytical treatment
ENTRY_YEAR / ENTRY_TERM,"Entry years span 2005–2025; terms include Fall, Spring, Summer",Entry cohort metadata; never interpreted as survey year
LEVEL_GRAD,"Observed codes: 1, 3, 4",Categorical code only; no unsupported degree labels


## 5. Recommended presentation arc

1. **Adoption:** 76% of valid respondents used generative AI; 28% used it weekly or daily.
2. **Use cases:** Research, coding, and brainstorming are the leading reported purposes.
3. **Guidance gap:** Self-reported understanding exceeds reported faculty/program guidance.
4. **Variation:** Purpose profiles differ meaningfully across adequately sized colleges.
5. **Action:** Pair university-wide principles with discipline-specific examples for coursework,
   research/writing, teaching, and code use.

### Limitations to say aloud

- Cross-sectional, self-reported, and unweighted.
- Item-level bases differ because of survey display logic and item nonresponse.
- Small college estimates are unstable; program-level publication would require suppression
  rules and a disclosure-risk review.
- `LEVEL_GRAD` labels and sampling/weighting metadata were not supplied.
- Results describe associations among respondents, not causal effects or population parameters.